# Step 1: Load and Inspect Your Datasets

In [ ]:
import pandas as pd

# Load the datasets (replace with your actual file paths)
try:
    df_sarcasm = pd.read_csv('/content/drive/MyDrive/NLP prj/sarcasm/train-balanced-sarcasm.csv')
    df_jokes = pd.read_csv('/content/drive/MyDrive/NLP prj/humor/shortjokes.csv')
except FileNotFoundError:
    print("Make sure your CSV files are in the correct folders!")
    exit()


# Inspect the sarcasm dataset
print("--- Sarcasm Dataset Info ---")
print(df_sarcasm.head())
print(df_sarcasm.info())
print("\nSarcasm label counts:")
print(df_sarcasm['label'].value_counts())

# Inspect the humor dataset
print("\n--- Humor Dataset Info ---")
print(df_jokes.head())
print(df_jokes.info())

--- Sarcasm Dataset Info ---
   label                                            comment     author  \
0      0                                         NC and NH.  Trumpbart   
1      0  You do know west teams play against west teams...  Shbshb906   
2      0  They were underdogs earlier today, but since G...   Creepeth   
3      0  This meme isn't funny none of the "new york ni...  icebrotha   
4      0                    I could use one of those tools.  cush2push   

            subreddit  score  ups  downs     date          created_utc  \
0            politics      2   -1     -1  2016-10  2016-10-16 23:55:23   
1                 nba     -4   -1     -1  2016-11  2016-11-01 00:24:10   
2                 nfl      3    3      0  2016-09  2016-09-22 21:45:37   
3  BlackPeopleTwitter     -8   -1     -1  2016-10  2016-10-18 21:03:47   
4  MaddenUltimateTeam      6   -1     -1  2016-12  2016-12-30 17:00:13   

                                      parent_comment  
0  Yeah, I get that argume

## Data preprocessing




In [ ]:
df_sarcasm.dropna(subset=['comment'], inplace=True)
df_jokes.dropna(subset=['Joke'], inplace=True)

df_sarcasm_processed = df_sarcasm[['comment', 'label']].copy()
df_jokes_processed = df_jokes[['Joke']].copy()

df_jokes_processed.rename(columns={'Joke': 'comment'}, inplace=True)
df_jokes_processed['label'] = 2

## Feature extraction




In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Combine the comment columns
combined_comments = pd.concat([df_sarcasm_processed['comment'], df_jokes_processed['comment']], ignore_index=True)

# Initialize TfidfVectorizer
tfidf_vectorizer = TfidfVectorizer()

# Fit and transform the combined comments
tfidf_features = tfidf_vectorizer.fit_transform(combined_comments)

# Inspect the shape of the resulting feature matrix
print("Shape of TF-IDF feature matrix:", tfidf_features.shape)

# Optionally inspect the vocabulary (first 100 words)
# print("\nVocabulary (first 100 words):", list(tfidf_vectorizer.vocabulary_.keys())[:100])

Shape of TF-IDF feature matrix: (1242428, 188382)


## Combine and split data



In [ ]:
from sklearn.model_selection import train_test_split

# Combine the labels
combined_labels = pd.concat([df_sarcasm_processed['label'], df_jokes_processed['label']], ignore_index=True)

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(tfidf_features, combined_labels, test_size=0.2, random_state=42)

# Verify the shapes of the resulting sets
print("Shape of X_train:", X_train.shape)
print("Shape of X_test:", X_test.shape)
print("Shape of y_train:", y_train.shape)
print("Shape of y_test:", y_test.shape)

Shape of X_train: (993942, 188382)
Shape of X_test: (248486, 188382)
Shape of y_train: (993942,)
Shape of y_test: (248486,)


## Model selection




In [ ]:
# Model Choice Reasoning

# 1. Logistic Regression:
# Advantages: Relatively simple, interpretable, and computationally efficient. Good baseline model for text classification.
# Disadvantages: May not capture complex non-linear relationships in the data as well as more complex models.

# 2. Simple Neural Network (e.g., with a few dense layers):
# Advantages: Can learn more complex patterns in the data than linear models. Can be more powerful than Logistic Regression for larger datasets.
# Disadvantages: Can be more computationally expensive to train. Requires careful tuning of hyperparameters.

# Preliminary Decision:
# Given the large size of the dataset and the potential for complex patterns in sarcasm and humor,
# a simple Neural Network is a good preliminary choice. It offers a balance between complexity and
# computational cost compared to very deep models like LSTMs or Transformers, while likely
# outperforming a simple Logistic Regression on this task. We can start with a basic neural network
# and consider more complex architectures later if needed.

## Model training

### Subtask:
Train the selected model (Simple Neural Network) on the training data (`X_train`, `y_train`).


In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

# Define the model architecture
model = Sequential([
    Dense(128, activation='relu', input_shape=(X_train.shape[1],)),
    Dense(64, activation='relu'),
    Dense(3, activation='softmax')  # 3 classes: 0 (not sarcastic), 1 (sarcastic), 2 (humor)
])

# Compile the model
model.compile(optimizer='adam',
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

# Train the model
history = model.fit(X_train, y_train,
                    epochs=1,  # You can adjust the number of epochs
                    batch_size=512, # You can adjust the batch size
                    validation_split=0.2) # Use 20% of training data for validation

# Display model summary
model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


1554/1554 ━━━━━━━━━━━━━━━━━━━━ 510s 326ms/step - accuracy: 0.6337 - loss: 0.7518 - val_accuracy: 0.6953 - val_loss: 0.6423


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 128)            │    24,113,024 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 3)              │           195 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 72,364,427 (276.05 MB)

 Trainable params: 24,121,475 (92.02 MB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 48,242,952 (184.03 MB)

## Model evaluation




In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Make predictions on the test set
y_pred_probs = model.predict(X_test)

# Convert predicted probabilities to class labels
y_pred = np.argmax(y_pred_probs, axis=1)

# Calculate evaluation metrics
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, average='weighted')
recall = recall_score(y_test, y_pred, average='weighted')
f1 = f1_score(y_test, y_pred, average='weighted')

# Print the metrics
print(f"Accuracy: {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1-score: {f1:.4f}")

7766/7766 ━━━━━━━━━━━━━━━━━━━━ 72s 9ms/step
Accuracy: 0.6952
Precision: 0.6976
Recall: 0.6952
F1-score: 0.6953


## Make predictions on new data



In [ ]:

new_comments = [
    "This is a truly amazing comment! I love it.", # Example of non-sarcastic/non-humorous
    "Wow, that was hilarious! I can't stop laughing.", # Example of humor
    "Oh, you're *so* funny. I'm dying of laughter here.", # Example of sarcasm
]


new_comments_features = tfidf_vectorizer.transform(new_comments)

# Make predictions using the trained model
new_predictions_probs = model.predict(new_comments_features)

# Convert predicted probabilities to class labels
# 0: not sarcastic, 1: sarcastic, 2: humor
new_predictions = np.argmax(new_predictions_probs, axis=1)

# Display the predictions
print("Predictions for new comments:")
for comment, prediction in zip(new_comments, new_predictions):
    label = "Not Sarcastic/Humorous" if prediction == 0 else ("Sarcastic" if prediction == 1 else "Humor")
    print(f"Comment: '{comment}' -> Predicted Label: {label}")

1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 678ms/step
Predictions for new comments:
Comment: 'This is a truly amazing comment! I love it.' -> Predicted Label: Not Sarcastic/Humorous
Comment: 'Wow, that was hilarious! I can't stop laughing.' -> Predicted Label: Sarcastic
Comment: 'Oh, you're *so* funny. I'm dying of laughter here.' -> Predicted Label: Sarcastic


# Task
Create a simple website using Flask where a user can input text, and the website will use the previously trained TensorFlow model and TF-IDF vectorizer to predict the category of the text. The website should display the prediction result to the user.

## Save the trained model

### Subtask:
Save the trained TensorFlow model and the fitted TF-IDF vectorizer to files so they can be loaded later in the web application.


**Reasoning**:
Save the trained TensorFlow model and the fitted TF-IDF vectorizer to files for later use in the web application.



In [ ]:
import pickle
import os

# Create a directory to save the model and vectorizer if it doesn't exist
save_dir = 'model_and_vectorizer'
if not os.path.exists(save_dir):
    os.makedirs(save_dir)

# Save the trained TensorFlow model
model_save_path = os.path.join(save_dir, 'sarcasm_humor_model.h5') # Using HDF5 format
model.save(model_save_path)
print(f"TensorFlow model saved to: {model_save_path}")

# Save the fitted TF-IDF vectorizer
vectorizer_save_path = os.path.join(save_dir, 'tfidf_vectorizer.pkl')
with open(vectorizer_save_path, 'wb') as f:
    pickle.dump(tfidf_vectorizer, f)
print(f"TF-IDF vectorizer saved to: {vectorizer_save_path}")

TensorFlow model saved to: model_and_vectorizer/sarcasm_humor_model.h5
TF-IDF vectorizer saved to: model_and_vectorizer/tfidf_vectorizer.pkl
